# AfriMedQA Llama 3.2 Fine-Tuning Pipeline

This notebook mirrors the project pipeline in `finetune_llm.py`. It is designed for Colab with GPU enabled and avoids duplicated model-loading blocks, synthetic-data defaults, and notebook-only syntax in Python scripts.


## 1. Install Dependencies

Run this once in Colab after selecting a GPU runtime.


In [ ]:
!pip install -q -r requirements-train.txt


## 2. Authenticate with Hugging Face

You need access to `meta-llama/Llama-3.2-3B-Instruct` and a Hugging Face token with read access.


In [ ]:
from huggingface_hub import notebook_login

notebook_login()


## 3. Import the Pipeline

The notebook uses the same functions as the command-line training script so behavior stays consistent.


In [ ]:
from finetune_llm import PipelineConfig, load_and_prepare_dataset, train

config = PipelineConfig(
    model_id="meta-llama/Llama-3.2-3B-Instruct",
    dataset_id="intronhealth/afrimedqa_v2",
    output_dir="outputs/afrimed_llama3_sft",
    final_adapter_dir="outputs/final_adapter",
)


## 4. Preview Formatted Training Data

This confirms the real AfriMedQA dataset is loading and formatting correctly before training starts.


In [ ]:
preview_config = PipelineConfig(max_train_samples=5)
preview_dataset = load_and_prepare_dataset(preview_config)
print(preview_dataset[0]["text"])


## 5. Smoke-Test Training

Run this small training pass first to confirm GPU, authentication, tokenization, QLoRA, and adapter saving all work.


In [ ]:
smoke_config = PipelineConfig(
    max_train_samples=16,
    num_train_epochs=0.01,
    save_steps=10,
    output_dir="outputs/smoke_test",
    final_adapter_dir="outputs/smoke_test/final_adapter",
)
train(smoke_config)


## 6. Full Training

After the smoke test succeeds, run full training. The final LoRA adapter will be saved to `outputs/final_adapter`.


In [ ]:
train(config)


## 7. Host the Streamlit Chatbot with Gemini

The hosted Streamlit app uses Gemini API directly, so no Hugging Face paid endpoint is required for the UI. Add your Gemini key to Streamlit secrets:

```toml
GEMINI_API_KEY = "your_gemini_api_key_here"
GEMINI_MODEL = "gemini-2.0-flash"
```
